<a href="https://colab.research.google.com/github/kushagarwal2910-lang/nlp-engiene-prototype/blob/main/prototype_ruxailab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import spacy
import json
from transformers import pipeline

nlp = spacy.load("en_core_web_sm")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

In [ ]:
def analyze_usability_text(raw_text):
    doc = nlp(raw_text)
    clean_text = " ".join([token.text for token in doc if not token.is_space])

    intent_labels = ["Complete Purchase", "Find Information", "Change Account Settings", "Navigate UI"]
    issue_labels = ["Unresponsive Button", "Confusing Navigation", "Slow Loading", "No Issue"]
    emotion_labels = ["Frustrated", "Confused", "Satisfied", "Neutral"]

    intent_result = classifier(clean_text, intent_labels)
    issue_result = classifier(clean_text, issue_labels)
    emotion_result = classifier(clean_text, emotion_labels)

    structured_data = {
        "raw_quote": raw_text,
        "analysis": {
            "primary_intent": intent_result['labels'][0],
            "perceived_issue": issue_result['labels'][0],
            "user_emotion": emotion_result['labels'][0]
        },
        "explainability_metrics": {
            "intent_confidence": round(intent_result['scores'][0], 3),
            "issue_confidence": round(issue_result['scores'][0], 3)
        }
    }

    return structured_data

In [ ]:
mock_transcripts = [
    "Where is the password reset page? I've been looking for five minutes and I just want to change it.",
    "Okay, I found the blog post, it loaded instantly. Looks good."
]
results = []
for quote in mock_transcripts:
    results.append(analyze_usability_text(quote))
print(json.dumps(results, indent=2))